# figure9.ipynb

Independently exports Figure 9 (Southern Ocean) from the final combined figure8.ipynb source, not the superseded figure9.ipynb.

Run the configuration cell first. Input data and fonts are not bundled.
See the repository README and data requirements before executing.


In [ ]:
from pathlib import Path
import sys as _sys

_repo_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / 'plot_config.py').is_file()
     and (p / 'config.example.json').is_file()),
    None,
)
if _repo_root is None:
    raise RuntimeError('Run this notebook from within the plotting repository.')
if str(_repo_root) not in _sys.path:
    _sys.path.insert(0, str(_repo_root))
from plot_config import configured_path


In [ ]:
import numpy as np  
import matplotlib.pyplot as plt  
import matplotlib.patches as patches 
import xarray as xr
import pandas as pd
import cartopy.crs as ccrs  
import cartopy.feature as cfeature
import cmaps
from cartopy.util import add_cyclic_point
import matplotlib.ticker as ticker
import matplotlib.dates as mdates

In [ ]:
fdir1 = configured_path('reccap_mask_file')
mask = xr.open_mfdataset(fdir1)
mask_ocean = mask.open_ocean

In [ ]:
fdir1 = configured_path('paired_mld_file')
ds = xr.open_mfdataset(fdir1)

time_new = pd.date_range(
    start='1982-01-01',
    periods=ds.sizes['time'],
    freq='MS'  # 每月开始
) + pd.Timedelta(days=14)  # 调整到每月中旬
ds = ds.assign_coords(time=time_new)
ds = ds.where(mask_ocean)
ds

In [ ]:
fdir2 = configured_path('iap_mld_file')
ds2 = xr.open_mfdataset(fdir2)

time_new = pd.date_range(
    start='1940-01-1',
    end='2023-12-15',
    freq='MS'  # 每月开始
) + pd.Timedelta(days=14)  # 调整到每月中旬
ds2 = ds2.assign_coords(time=time_new)

ds2 = ds2.sel(time=slice('1993','2023'))
ds2 = ds2.where(mask_ocean)
ds2

In [ ]:
fdir3 = configured_path('oras_mld_pattern')
ds3 = xr.open_mfdataset(fdir3)
ds3 = ds3.resample(time='1ME').mean(dim='time')

time_new = pd.date_range(
    start='1993-01-1',
    end='2023-12-15',
    freq='MS'  # 每月开始
) + pd.Timedelta(days=14)  # 调整到每月中旬
ds3 = ds3.assign_coords(time=time_new)
ds3 = ds3.where(mask_ocean)
ds3

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import font_manager
from pathlib import Path
import numpy as np
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cmaps
import matplotlib.ticker as ticker
from matplotlib.colors import BoundaryNorm  # 关键缺失库

times_font_dir = Path(configured_path('font_dir'))
times_font_files = [times_font_dir / 'Times New Roman.ttf', times_font_dir / 'Times New Roman Bold.ttf']
for font_file in times_font_files:
    if not font_file.is_file():
        raise FileNotFoundError(f'Required Times New Roman font not found: {font_file}')
    font_manager.fontManager.addfont(str(font_file))
resolved_times = Path(font_manager.findfont('Times New Roman', fallback_to_default=False)).resolve()
if resolved_times not in {font.resolve() for font in times_font_files}:
    raise RuntimeError(f'Times New Roman resolved to an unexpected file: {resolved_times}')
mpl.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman'],
    'mathtext.fontset': 'custom',
    'mathtext.rm': 'Times New Roman',
    'mathtext.it': 'Times New Roman:italic',
    'mathtext.bf': 'Times New Roman:bold',
    'axes.unicode_minus': False,
})
# 自定义色板
cmap1 = cmaps.precip_11lev[1:]
cmap2 = cmap1.interp(10)

# ====================== 1. 筛选区域 + 时间范围 + 提取JJA夏季多年气候态 ======================
def calc_jja_clim(ds_in):
    # 1. 时间范围1993-2022，40°S以南
    ds_sub = ds_in.sel(time=slice("1993", "2023"))\
                  .where((ds_in.lat >= -90) & (ds_in.lat <= -40), drop=True)
    # 2. 只保留每年6、7、8月 JJA
    ds_jja = ds_sub.where(ds_sub.time.dt.season == "JJA", drop=True)
    # 3. 逐年夏季平均（每年JJA三个月均值）
    ds_jja_year = ds_jja.groupby("time.year").mean("time", skipna=True)
    # 4. 多年夏季气候态平均
    ds_jja_clim = ds_jja_year.mean("year", skipna=True)
    return ds_jja_clim

# 数据集分别计算JJA气候态
ds_region = calc_jja_clim(ds)
ds2_region = calc_jja_clim(ds2.mld)
ds3_region = calc_jja_clim(ds3.MLD_sigma0_030)

# ====================== 2. 提取JJA多年平均MLD场 ======================
mld_pred = ds_region["MLD_pred"].compute()
mld_true = ds_region["MLD_true"].compute()
mld_iap  = ds2_region.compute()
mld_oras = ds3_region.compute()

# 绘图数据与子标题
plot_info = [
    (mld_pred, "SJTU-ViT"),
    (mld_true, "Obs"),
    (mld_iap, "IAP"),
    (mld_oras, "ORAS5")
]

# 与 figure9 一致的南大洋区域平均时间序列（2010–2020）
ts_ds = ds.sel(lat=slice(-60, -50), lon=slice(230, 290), time=slice('2010', '2020'))
ts_iap = ds2.mld.sel(lat=slice(-60, -50), lon=slice(230, 290), time=slice('2010', '2020'))
ts_oras = ds3.MLD_sigma0_030.sel(lat=slice(-60, -50), lon=slice(230, 290), time=slice('2010', '2020'))
ts_series = [
    ("SJTU-ViT", ts_ds.MLD_pred.mean(["lat", "lon"], skipna=True).compute(), "#d62728"),
    ("Obs", ts_ds.MLD_true.mean(["lat", "lon"], skipna=True).compute(), "#1f77b4"),
    ("IAP", ts_iap.mean(["lat", "lon"], skipna=True).compute(), "#17becf"),
    ("ORAS5", ts_oras.mean(["lat", "lon"], skipna=True).compute(), "#2ca02c"),
]

# 全局统一色标范围
all_data_stack = np.concatenate([
    mld_pred.values.flatten(),
    mld_true.values.flatten(),
    mld_iap.values.flatten(),
    mld_oras.values.flatten()
])
all_data_stack = all_data_stack[~np.isnan(all_data_stack)]  # 剔除 NaN
vmin = float(np.min(all_data_stack))
vmax = float(np.max(all_data_stack))

# ====================== 构造非线性色阶 ======================
nonlinear_levels = [10, 20, 40, 65, 100, 150, 220, 300, 400]
cmap1 = cmaps.precip_11lev[1:]
cmap2 = cmap1.interp(20)  # 足够多颜色给分段使用
# 核心：BoundaryNorm 强制色板严格按你的边界分层，不会渐变融合
norm = BoundaryNorm(nonlinear_levels, ncolors=cmap2.N, extend="max")

# ====================== 3. 2×2地图 + 底部时间序列 ======================
fig = plt.figure(figsize=(7.5, 9.2), dpi=300)
gs = fig.add_gridspec(
    3, 3, width_ratios=[1, 1, 0.075], height_ratios=[1, 1, 0.72],
    left=0.07, right=0.95, bottom=0.07, top=0.97, wspace=0.06, hspace=0.15
)
map_projection = ccrs.SouthPolarStereo(central_longitude=0)
axes_flat = [
    fig.add_subplot(gs[0, 0], projection=map_projection),
    fig.add_subplot(gs[0, 1], projection=map_projection),
    fig.add_subplot(gs[1, 0], projection=map_projection),
    fig.add_subplot(gs[1, 1], projection=map_projection),
]

for idx, ((data, title), ax) in enumerate(zip(plot_info, axes_flat)):
    # 限定范围：全球经度，40°S~90°S
    ax.set_extent([-180, 180, -90, -40], crs=ccrs.PlateCarree())

    # 绘制圆形边界，做成和参考图一样的圆形南极俯视图
    import matplotlib.patches as mpatches
    theta = np.linspace(0, 2*np.pi, 300)
    center, radius = [0.5, 0.5], 0.5
    verts = np.vstack([np.cos(theta)*radius + center[0],
                       np.sin(theta)*radius + center[1]]).T
    circle_path = mpatches.Path(verts)
    circle_patch = mpatches.PathPatch(circle_path, transform=ax.transAxes,
                                      facecolor="none", edgecolor="black", lw=0.6)
    ax.add_patch(circle_patch)
    ax.set_boundary(circle_path, transform=ax.transAxes)

    # 填色等值线：使用自定义非线性levels
    cs = ax.contourf(
        data.lon, data.lat, data,
        levels=nonlinear_levels, 
        norm=norm,
        cmap=cmap2,
        transform=ccrs.PlateCarree(),
        #extend="both"
    )

    # 陆地、海岸线
    ax.add_feature(cfeature.COASTLINE, lw=0.5)
    ax.add_feature(cfeature.LAND, facecolor="lightgray", zorder=3)

    # 极地网格线
    gl = ax.gridlines(draw_labels=False, lw=0.3, alpha=0.7, color="black")
    gl.ylocator = plt.MultipleLocator(20)   # 40S/60S/80S
    gl.xlocator = plt.MultipleLocator(30)   # 每30°一条经线
    # ========== 新增：区域标线框 ==========
    from matplotlib.patches import Rectangle
    lon0, lon1 = 230, 290
    lat0, lat1 = -60, -50
    box = Rectangle(
        (lon0, lat0), lon1-lon0, lat1-lat0,
        transform=ccrs.PlateCarree(),
        fill=False, edgecolor="crimson", lw=1.1, linestyle="--"
    )
    ax.add_patch(box)
    ax.set_title(title, fontsize=15, loc="center", pad=4)
    ax.text(0.0, 1.04, f"({chr(97 + idx)})", transform=ax.transAxes,
            ha="left", va="bottom", fontsize=13)

# 地图统一色条
cbar_ax = fig.add_subplot(gs[:2, 2])
cbar_pos = cbar_ax.get_position()
cbar_ax.set_position([cbar_pos.x0, cbar_pos.y0 + 0.08 * cbar_pos.height,
                      cbar_pos.width, 0.84 * cbar_pos.height])
cbar = fig.colorbar(cs, cax=cbar_ax,
                    ticks=nonlinear_levels,
                    format=ticker.ScalarFormatter(),
                    spacing="uniform")
cbar.set_label("Climatological JJA MLD (m)", fontsize=13)
cbar.ax.tick_params(labelsize=12)

# 底部区域平均时间序列
ax_ts = fig.add_subplot(gs[2, :])
for label, series, color in ts_series:
    ax_ts.plot(series.time.values, series.values, color=color, lw=1.5, label=label)
ax_ts.legend(loc="upper left", ncol=4, frameon=True, fontsize=10)
ax_ts.set_title("Regional Averaged MLD (50–60°S, 230–290°E)", fontsize=13, pad=4)
ax_ts.set_xlabel("Date (YYYY)", fontsize=12)
ax_ts.set_ylabel("MLD (m)", fontsize=12)
ax_ts.set_ylim(20, 470)
ax_ts.xaxis.set_major_locator(mdates.YearLocator(2))
ax_ts.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax_ts.grid(alpha=0.3, linewidth=0.4)
ax_ts.tick_params(axis="both", labelsize=10.5)
ax_ts.text(0.0, 1.04, "(e)", transform=ax_ts.transAxes,
           ha="left", va="bottom", fontsize=13)

output_dir = Path(configured_path('output_dir'))
output_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(output_dir / 'figure9.png',
            dpi=600, bbox_inches='tight', pad_inches=0)
plt.show()